In [ ]:


# --- SELEZIONE E RINOMINA COLONNE ---
df_comuni = df_comuni[[
    "Codice Comune (numerico)",
    "Codice Comune (alfanumerico)",
    "Comune",
    "Codice Ripartizione geografica",
    "Codice Regione",
    "Codice Provincia/Uts",
    "Sigla automobilistica",
    "Superficie (Kmq)",
    "Popolazione residente",
    "Capoluogo di Provincia/Uts",
    "Capoluogo di Regione"
]].copy()

df_comuni.columns = [
    "IDN_Comune",
    "IDA_Comune",
    "Comune",
    "ID_Ripartizione_Geografica",
    "ID_Regione",
    "ID_Provincia_Uts",
    "Sigla_Auto",
    "Superficie_Kmq",
    "Pop_Residente",
    "IS_Capoluogo_Provincia_Uts",
    "IS_Capoluogo_Regione"
]

# --- CONVERSIONE SUPERFICIE DA VIRGOLA A PUNTO ---
# Il file SITUAS usa la virgola come separatore decimale (formato italiano)
df_comuni["Superficie_Kmq"] = (
    df_comuni["Superficie_Kmq"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# --- FIX COMUNE "NONE" (IDN_Comune = 1168, TO) ---
# Il comune "None" (Torino) viene letto da pandas come NaN
# perché "None" è anche una keyword Python — fix manuale
df_comuni.loc[df_comuni["IDN_Comune"] == 1168, "Comune"] = "None"

# --- ANNO DI RIFERIMENTO ---
df_comuni["Anno_Riferimento_SITUAS"] = 2024

# --- VERIFICA FINALE ---
print("\n=== VERIFICA FINALE ===")
print(f"Shape: {df_comuni.shape}")
print(f"\nNull per colonna:")
print(df_comuni.isnull().sum())
print(f"\nDtypes:")
print(df_comuni.dtypes)

# --- EXPORT ---
df_comuni.to_csv("../data/situas_clean.csv", index=False)
print("\n✅ Salvato: data/situas_clean.csv")

# IMPORTAZIONE LIBRERIE E DATAFRAME

In [1]:
import pandas as pd

In [4]:
df_2024 = pd.read_csv("../raw_data/comuni_2024-12-31.csv", sep=";")
df_2026 = pd.read_csv("../raw_data/comuni_2026-06-21.csv", sep=";")

In [5]:
print(f"File 2024: {df_2024.shape}")
print(f"File 2026: {df_2026.shape}")

File 2024: (7896, 17)
File 2026: (7894, 17)


# INNER MERGE

Eseguo inner join perche' voglio tenere solo comuni che al 2026 esistono ancora e su cui posso effettuare analisi statistiche grazie allo storico presente al 2024 (sul periodo 2001-2024).
La sardegna (regione 20) verra' inclusa separatamente poiche' presenta codici differenti tra 2024 e 2026 per cambio numerazione e verrebbe esclusa dall'inner join.

## REGIONI 1-19

In [7]:
df_119_2024 = df_2024[df_2024["Codice Regione"] != 20]
df_119_2026 = df_2026[df_2026["Codice Regione"] != 20]

df_119_merge2426 = df_119_2024.merge(df_119_2026[["Codice Comune (numerico)"]], on="Codice Comune (numerico)", how="inner")

print("Comuni regioni 1-19 (inner merge): ", len(df_119_merge2426))

Comuni regioni 1-19 (inner merge):  7516


## REGIONE 20 (SARDEGNA)

In [8]:
df_sardegna = df_2024[df_2024["Codice Regione"] == 20].copy()

print("Comuni Sardegna (2024): ", len(df_sardegna))

Comuni Sardegna (2024):  377


## CONCATENAZIONE REGIONI 1-19 E 20

In [ ]:
df_comuni = pd.concat([df_119_merge2426, df_sardegna], ignore_index=True)

print("Totale comuni: ", len(df_comuni))
print(f"Regioni presenti: {sorted(df_comuni['Codice Regione'].unique())}")